# Demo - Two Coordinated Sessions End to End
**Day 1 - Session 1, Topic 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-two-session-orchestration.ipynb)

**Goal:** Run two isolated worktree sessions through commit, handoff, dependency-ordered merge, and a real acceptance gate, then resolve a genuine merge conflict.

This is the full orchestration loop with nothing simulated. Each session gets its own Git worktree and branch, edits only what its card owns, runs its own focused tests, and writes a handoff recording its real commit and real result. Integration merges in dependency order and pauses at one human approval point.

> Requires Git. Colab has no global Git identity, so the sandbox sets its own.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [1]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {{REPO_URL}} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

Course root: /Users/kangs/code/github/agent-orchestration-companion
Git: git version 2.50.1 (Apple Git-155)


## 2. Open two isolated sessions

Each session is a real `git worktree` on its own branch, created from the approved base.


In [2]:
import json
from pathlib import Path

from demo_support import assert_true, git, heading, run_tests, sandbox, show_evidence

SESSIONS = (
    {"name": "email", "branch": "agent/email", "file": Path("channels") / "email.py",
     "edit": ("your order shipped", "your order has shipped"), "tests": "tests.test_email"},
    {"name": "sms", "branch": "agent/sms", "file": Path("channels") / "sms.py",
     "edit": ("MAX_LENGTH = 160", "MAX_LENGTH = 150"),
     "tests": "tests.test_security.ChannelSecurityTest.test_sms_normalizes_control_whitespace"},
)
MERGE_ORDER = ("agent/email", "agent/sms")
APPROVED_ORDER = 'CHANNEL_ORDER = ("email", "sms")'

sandbox_context = sandbox()
WORK, BASE = sandbox_context.__enter__()
show_evidence("approved base commit", BASE)

heading("Open two isolated sessions")
WORKTREES = {}
for session in SESSIONS:
    worktree = WORK.parent / f"work-{session['name']}"
    git("worktree", "add", "-q", "-b", session["branch"], str(worktree), BASE, cwd=WORK)
    WORKTREES[session["name"]] = worktree
    show_evidence(f"{session['branch']} worktree", worktree.name)
show_evidence("git worktree count", len(git("worktree", "list", cwd=WORK).splitlines()))

  approved base commit       301b320

Open two isolated sessions
--------------------------
  agent/email worktree       work-email
  agent/sms worktree         work-sms
  git worktree count         3


## 3. Each session works inside its own boundary

Edit the owned file, run the focused test, commit, then record real evidence in the handoff.


In [3]:
HANDOFFS = {}
heading("Each session works inside its own boundary")
for session in SESSIONS:
    worktree = WORKTREES[session["name"]]
    path = worktree / session["file"]
    path.write_text(path.read_text().replace(*session["edit"]))
    passed, _ = run_tests(session["tests"], worktree)
    git("add", "-A", cwd=worktree)
    git("commit", "-q", "-m", f"Complete {session['branch']}", cwd=worktree)
    commit = git("rev-parse", "--short", "HEAD", cwd=worktree)
    changed = git("diff", "--name-only", "HEAD~1..HEAD", cwd=worktree).splitlines()

    handoff_path = worktree / "handoffs" / f"{session['name']}.json"
    handoff = json.loads(handoff_path.read_text())
    handoff.update({
        "base_commit": BASE,
        "implementation_commit": commit,
        "changed_files": sorted(changed),
        "verification": {
            "command": f"python3 -m unittest {session['tests']}",
            "status": "passed" if passed else "failed",
            "summary": "Focused acceptance command run inside the session worktree.",
        },
    })
    handoff_path.write_text(json.dumps(handoff, indent=2) + "\n")
    git("add", "-A", cwd=worktree)
    git("commit", "-q", "-m", f"Record {session['name']} handoff", cwd=worktree)
    HANDOFFS[session["name"]] = handoff

    show_evidence(f"{session['name']} commit", commit)
    show_evidence(f"{session['name']} changed", ", ".join(changed))
    show_evidence(f"{session['name']} focused test", "passed" if passed else "FAILED")

print()
print(json.dumps(HANDOFFS["email"], indent=2))


Each session works inside its own boundary
------------------------------------------
  email commit               18424e0
  email changed              channels/email.py
  email focused test         passed
  sms commit                 b2445d1
  sms changed                channels/sms.py
  sms focused test           passed

{
  "task_id": "email-renderer",
  "base_commit": "301b320",
  "implementation_commit": "18424e0",
  "changed_files": [
    "channels/email.py"
  ],
  "decisions": [
    "Used standard-library HTML escaping for untrusted event values."
  ],
  "verification": {
    "command": "python3 -m unittest tests.test_email",
    "status": "passed",
    "summary": "Focused acceptance command run inside the session worktree."
  },
  "unresolved_risks": [],
  "next_checkpoint": "Human review before integration."
}


## 4. Prove isolation before any merge

`main` must not see either session's edits yet. That is what the worktree buys.


In [4]:
MAIN_EMAIL = (WORK / "channels" / "email.py").read_text()

heading("Isolation proof before any merge")
show_evidence("main has email wording change", SESSIONS[0]["edit"][1] in MAIN_EMAIL)
show_evidence("main still at base", git("rev-parse", "--short", "HEAD", cwd=WORK) == BASE)


Isolation proof before any merge
--------------------------------
  main has email wording change False
  main still at base         True


## 5. Integrate in dependency order with one approval point

Merge `agent/email` first, review both handoffs, then merge `agent/sms` and run the full suite.


In [5]:
heading("Integrate in dependency order with one approval point")
for index, branch in enumerate(MERGE_ORDER, start=1):
    if index == 2:
        evidence_ok = all(
            handoff["verification"]["status"] == "passed" for handoff in HANDOFFS.values()
        )
        show_evidence("human approval point", "review both handoffs before merge 2")
        show_evidence("handoff evidence complete", evidence_ok)
    git("merge", "-q", "--no-edit", branch, cwd=WORK)
    show_evidence(f"merge {index}", f"{branch} -> main")

COMBINED_OK, combined_output = run_tests("discover", WORK)
ran = [line for line in combined_output.splitlines() if line.startswith("Ran ")]
show_evidence("combined suite", "passed" if COMBINED_OK else "FAILED")
show_evidence("tests executed", ran[0] if ran else "(unknown)")


Integrate in dependency order with one approval point
-----------------------------------------------------
  merge 1                    agent/email -> main
  human approval point       review both handoffs before merge 2
  handoff evidence complete  True
  merge 2                    agent/sms -> main
  combined suite             passed
  tests executed             Ran 6 tests in 0.000s


## 6. Resolve a real merge conflict on the shared router

Two branches change the same line of `router.py`, so Git genuinely conflicts.


In [6]:
git("checkout", "-q", "-b", "conflict-practice", cwd=WORK)
router = WORK / "router.py"
router.write_text(router.read_text().replace(APPROVED_ORDER, 'CHANNEL_ORDER = ("sms", "email")'))
git("commit", "-q", "-am", "Reorder channels on a practice branch", cwd=WORK)
git("checkout", "-q", "main", cwd=WORK)
router.write_text(router.read_text().replace(
    APPROVED_ORDER, f"{APPROVED_ORDER}  # order approved at integration"
))
git("commit", "-q", "-am", "Annotate the approved channel order", cwd=WORK)

heading("Resolve a real merge conflict on the shared router")
conflicted = git("merge", "conflict-practice", cwd=WORK, check=False)
STATUS = git("status", "--porcelain", cwd=WORK)
show_evidence("merge exit", "conflict" if "UU" in STATUS else "clean")
show_evidence("git reported", [line for line in conflicted.splitlines() if "CONFLICT" in line])
show_evidence("conflicted path", [line for line in STATUS.splitlines() if line.startswith("UU")])

git("checkout", "--ours", "router.py", cwd=WORK)
git("add", "router.py", cwd=WORK)
git("commit", "-q", "-m", "Keep approved channel order", cwd=WORK)
FINAL_ORDER = APPROVED_ORDER in (WORK / "router.py").read_text()
FINAL_OK, _ = run_tests("discover", WORK)
show_evidence("approved order preserved", FINAL_ORDER)
show_evidence("suite after resolution", "passed" if FINAL_OK else "FAILED")


Resolve a real merge conflict on the shared router
--------------------------------------------------
  merge exit                 conflict
  git reported               ['CONFLICT (content): Merge conflict in router.py']
  conflicted path            ['UU router.py']
  approved order preserved   True
  suite after resolution     passed


## 7. Verify the evidence and clean up

Remove the worktrees so the temporary repository tears down cleanly.


In [7]:
heading("Evidence checks")
assert_true(SESSIONS[0]["edit"][1] not in MAIN_EMAIL,
            "worktrees kept session edits out of main before merge")
assert_true(all(h["implementation_commit"] != h["base_commit"] for h in HANDOFFS.values()),
            "each handoff recorded a real commit distinct from the base")
assert_true(COMBINED_OK, "the full suite passed on the merged tree")
assert_true("UU router.py" in STATUS, "Git produced a genuine merge conflict")
assert_true(FINAL_ORDER and FINAL_OK, "resolution kept the approved order and stayed green")

for session in SESSIONS:
    git("worktree", "remove", "--force", str(WORKTREES[session["name"]]), cwd=WORK)
sandbox_context.__exit__(None, None, None)

print("\nTakeaway: Isolated worktrees, durable handoffs, ordered merges, and one")
print("approval point turn parallel sessions into a verifiable integration.")


Evidence checks
---------------
  [verified] worktrees kept session edits out of main before merge
  [verified] each handoff recorded a real commit distinct from the base
  [verified] the full suite passed on the merged tree
  [verified] Git produced a genuine merge conflict
  [verified] resolution kept the approved order and stayed green

Takeaway: Isolated worktrees, durable handoffs, ordered merges, and one
approval point turn parallel sessions into a verifiable integration.


### Expected output

- Three worktrees listed: `main` plus one per session.
- Each session commits only its own file; the printed handoff JSON carries a
  real `implementation_commit` and a `passed` verification status.
- Isolation: `main has email wording change` is `False` and `main still at
  base` is `True`.
- Two merges in order with an approval point between them, then
  `Ran 6 tests` passing on the merged tree.
- Git reports `CONFLICT (content): Merge conflict in router.py` and
  `UU router.py`; after resolution the approved order survives and the suite
  is still green.
- Five `[verified]` lines, then the takeaway.
